<a href="https://colab.research.google.com/github/rager1904/OpenBioDesign/blob/main/platform/colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenBioDesign - ML-Powered Protein Binder Design Platform

This notebook runs the complete OpenBioDesign platform with **real ML models** on Google Colab free tier.

### Models Used
- **ESM2-650M**: Protein language model for sequence analysis, binding site detection, and candidate generation
- **ESMFold**: Single-sequence structure prediction with pLDDT confidence scores

### What This Platform Does
1. Analyzes protein targets to identify binding sites
2. Generates candidate protein binder sequences
3. Predicts 3D structures of candidates
4. Scores mutations for stability effects
5. Provides explainable AI predictions with confidence metrics

### Remote Access
Both the frontend and backend are exposed via localtunnel URLs so you can access the platform from **any device**.

## Cell 1: Install Dependencies

In [ ]:
!pip install fastapi uvicorn sqlalchemy pydantic pydantic-settings python-multipart
!pip install transformers torch --quiet
!pip install numpy scipy httpx
!pip install localtunnel

!apt-get update -qq && apt-get install -y nodejs npm -qq

print('Dependencies installed successfully!')

## Cell 2: Clone Repository

In [ ]:
import os

!git clone https://github.com/rager1904/OpenBioDesign.git /content/OpenBioDesign 2>/dev/null || echo 'Using existing repo'

os.chdir('/content/OpenBioDesign/platform/backend')

print(f'Current directory: {os.getcwd()}')

## Cell 3: Check GPU and Load ESM2 Model

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {vram:.1f} GB')
    print('GPU is available and ready!')
else:
    print('WARNING: No GPU detected. Models will run on CPU (slower).')
    print('Go to Runtime > Change runtime type > GPU')

print('\nLoading ESM2-650M model...')
from transformers import AutoModelForMaskedLM, AutoTokenizer

ESM2_MODEL = 'facebook/esm2_t33_650M_UR50D'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL)
esm2_model = AutoModelForMaskedLM.from_pretrained(ESM2_MODEL)
esm2_model = esm2_model.to(device)
esm2_model.eval()

print(f'ESM2 loaded on {device}')
print(f'Model parameters: {sum(p.numel() for p in esm2_model.parameters()) / 1e6:.0f}M')

## Cell 4: Load ESMFold Model

In [ ]:
print('Loading ESMFold model...')

!pip install fair-esm --quiet
import esm

esmfold_model, _ = esm.pretrained.esmfold_v1()
esmfold_model = esmfold_model.to(device)
esmfold_model.eval()
print('ESMFold loaded successfully!')

## Cell 5: Initialize Backend with ML Agents

In [ ]:
import sys
import os

sys.path.insert(0, os.getcwd())

from openbiodesign.infrastructure.esm2_client import ESM2Client

class PatchedESM2Client(ESM2Client):
    def __init__(self):
        self.model = esm2_model
        self.tokenizer = tokenizer
        self.device = device
        self._model_loaded = True

ESM2Client._instance = PatchedESM2Client()
print('ESM2 client patched with pre-loaded model')

from openbiodesign.infrastructure.esmfold_client import ESMFoldClient

class PatchedESMFoldClient(ESMFoldClient):
    def __init__(self):
        self.model = esmfold_model
        self.device = device
        self._model_loaded = True

ESMFoldClient._instance = PatchedESMFoldClient()
print('ESMFold client patched with pre-loaded model')

In [ ]:
import subprocess
import time

!pkill -f uvicorn 2>/dev/null || true
time.sleep(1)

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'openbiodesign.main:app', '--host', '0.0.0.0', '--port', '8080'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

import httpx
try:
    response = httpx.get('http://localhost:8080/api/v1/health')
    print('Backend server is running!')
    print(f'API Health: {response.json()}')
except Exception as e:
    print(f'Server check failed: {e}')
    print('Checking server logs...')
    _, stderr = server.communicate(timeout=1)
    print(stderr.decode()[:500] if stderr else 'No errors')

## Cell 6: Expose Backend & Build Frontend

Creates a public URL for the backend, then builds the frontend pointing to that URL.

In [ ]:
import subprocess
import time
import re

# --- Create backend tunnel (port 8080) ---
print('Creating backend tunnel...')
backend_tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8080'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)

# Capture the backend tunnel URL
backend_url = ''
if backend_tunnel.stdout:
    for _ in range(10):
        line = backend_tunnel.stdout.readline()
        if 'loca.lt' in line or 'https' in line:
            backend_url = line.strip()
            break
        time.sleep(0.5)

if not backend_url:
    # Fallback: read whatever is available
    import select
    backend_url = backend_tunnel.stdout.readline().strip()

print(f'Backend tunnel URL: {backend_url}')

# Build the full API base URL for the frontend
# Remove trailing slash if present, then append /api/v1
api_base = backend_url.rstrip('/') + '/api/v1'
print(f'Frontend will call API at: {api_base}')

# --- Build frontend with the backend tunnel URL ---
frontend_dir = '/content/OpenBioDesign/platform/frontend'
os.chdir(frontend_dir)

!npm install --silent 2>/dev/null

# Build with NEXT_PUBLIC_API_BASE_URL so the frontend knows where the backend is
env = os.environ.copy()
env['NEXT_PUBLIC_API_BASE_URL'] = api_base

print('\nBuilding frontend with backend tunnel URL...')
build_result = subprocess.run(
    ['npm', 'run', 'build'],
    env=env,
    capture_output=True,
    text=True
)

if build_result.returncode != 0:
    print(f'Build warning: {build_result.stderr[:500]}')
else:
    print('Frontend built successfully!')

## Cell 7: Start Frontend & Expose via Tunnel

Starts the frontend and creates a public URL for it.

In [ ]:
import subprocess
import time

# Start the frontend server
env = os.environ.copy()
env['NEXT_PUBLIC_API_BASE_URL'] = api_base

frontend_server = subprocess.Popen(
    ['npx', 'next', 'start', '-H', '0.0.0.0', '-p', '3000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env
)
time.sleep(3)

# --- Create frontend tunnel (port 3000) ---
print('Creating frontend tunnel...')
frontend_tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '3000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)

# Capture frontend tunnel URL
frontend_url = ''
if frontend_tunnel.stdout:
    for _ in range(10):
        line = frontend_tunnel.stdout.readline()
        if 'loca.lt' in line or 'https' in line:
            frontend_url = line.strip()
            break
        time.sleep(0.5)

if not frontend_url:
    frontend_url = frontend_tunnel.stdout.readline().strip()

print()
print('=' * 60)
print('  OPENBIODESIGN - REMOTE ACCESS URLS')
print('=' * 60)
print()
print(f'  Frontend UI:   {frontend_url}')
print(f'  Backend API:   {backend_url}')
print()
print('  Open the Frontend URL on any device (phone, tablet, laptop).')
print('  Share these URLs with collaborators.')
print()
print('  NOTE: On first visit, localtunnel shows a "Click to Continue"')
print('  page. Just click through it to proceed.')
print('=' * 60)

## Cell 8: Demo - Protein Analysis with ESM2

In [ ]:
import httpx
import json

TARGET_SEQUENCE = (
    'MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIR'
    'GNMYYENSYALAVLSNINDFNATHTKKEGYGTVIKWVPESGALKKETXAAFKKEGYGTVIKWVPESGALKKETXAAFK'
)

print('Target: EGFR (Epidermal Growth Factor Receptor)')
print(f'Sequence length: {len(TARGET_SEQUENCE)} residues\n')

response = httpx.post(
    'http://localhost:8080/api/v1/esm2/detect-binding-sites',
    json={'sequence': TARGET_SEQUENCE, 'top_k': 8},
    timeout=60.0
)

if response.status_code == 200:
    result = response.json()
    print('=== ESM2 Binding Site Detection ===')
    print(f'Binding site residues (1-indexed): {result["residue_positions_1indexed"]}')
    print(f'Confidence: {result["confidence"]:.4f}')
    print(f'Method: {result["method"]}')
    print(f'\nAttention scores: {[f"{s:.3f}" for s in result["attention_scores"]]}')
else:
    print(f'Error: {response.status_code}')
    print(response.text)

In [ ]:
response = httpx.post(
    'http://localhost:8080/api/v1/esm2/score-sequence',
    json={'sequence': TARGET_SEQUENCE},
    timeout=60.0
)

if response.status_code == 200:
    result = response.json()
    print('=== ESM2 Sequence Fitness Score ===')
    print(f'Mean log-likelihood: {result["mean_log_likelihood"]:.4f}')
    print(f'Interpretation: {result["interpretation"]}')
    print(f'\nPer-residue scores (first 20):')
    for i, score in enumerate(result['per_residue_log_likelihood'][:20]):
        print(f'  Position {i+1}: {score:.4f}')
else:
    print(f'Error: {response.status_code}')

## Cell 9: Demo - Candidate Generation

In [ ]:
response = httpx.post(
    'http://localhost:8080/api/v1/workflows/binder-design',
    json={
        'project_id': 'colab-demo-project',
        'target': {
            'name': 'EGFR',
            'sequence': TARGET_SEQUENCE,
            'organism': 'Homo sapiens'
        },
        'hypothesis': 'Design a protein binder that targets the EGFR kinase domain for potential cancer therapy applications',
        'requested_candidates': 3,
        'random_seed': 42
    },
    headers={'Authorization': 'Bearer scientist'},
    timeout=120.0
)

if response.status_code == 200:
    result = response.json()
    print('=== Binder Design Results ===')
    print(f'Target: {result["target"]["name"]}')
    print(f'Binding sites detected: {len(result["binding_sites"])}')
    print(f'Candidates generated: {len(result["candidates"])}')

    for i, candidate in enumerate(result['candidates']):
        print(f'\n--- Candidate {i+1} ---')
        print(f'Scaffold: {candidate["scaffold_id"]}')
        print(f'Sequence: {candidate["sequence"][:50]}...')
        print(f'Binding score: {candidate["binding_score"]}')
        print(f'Stability score: {candidate["stability_score"]}')
        print(f'Manufacturability: {candidate["manufacturability_score"]}')

        for metric in candidate['confidence_metrics'][:2]:
            print(f'  {metric["name"]}: {metric["value"]:.3f}')
else:
    print(f'Error: {response.status_code}')
    print(response.text[:500])

## Cell 10: Demo - Structure Prediction with ESMFold

In [ ]:
if 'result' in locals() and result['candidates']:
    candidate_seq = result['candidates'][0]['sequence']
    print(f'Predicting structure for candidate sequence ({len(candidate_seq)} residues)...\n')

    response = httpx.post(
        'http://localhost:8080/api/v1/esmfold/predict',
        json={'sequence': candidate_seq},
        timeout=120.0
    )

    if response.status_code == 200:
        structure = response.json()
        print('=== ESMFold Structure Prediction ===')
        print(f'Sequence length: {structure["sequence_length"]} residues')
        print(f'Mean pLDDT: {structure["mean_plddt"]:.2f}')
        print(f'Confidence: {structure["confidence_classification"]}')
        print(f'Interpretation: {structure["interpretation"]}')

        print('\nConfidence breakdown:')
        stats = structure['confidence_summary']
        print(f'  Confident (>90 pLDDT): {stats["confident_pct"]:.1f}%')
        print(f'  Good (70-90): {stats["good_pct"]:.1f}%')
        print(f'  Low (50-70): {stats["low_pct"]:.1f}%')
        print(f'  Very low (<50): {stats["very_low_pct"]:.1f}%')

        with open('predicted_structure.pdb', 'w') as f:
            f.write(structure['pdb_content'])
        print(f'\nPDB file saved to: predicted_structure.pdb')
    else:
        print(f'Error: {response.status_code}')
else:
    print('No candidates available. Run Cell 9 first.')

## Cell 11: Demo - Mutation Analysis

In [ ]:
test_sequence = 'ACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRSTVWY'

print('=== ESM2 Mutation Effect Prediction ===\n')
print(f'Test sequence: {test_sequence}\n')

mutations_to_test = [
    (5, 'A', 'D'),
    (10, 'K', 'E'),
    (15, 'P', 'G'),
]

for pos, wt, mut in mutations_to_test:
    response = httpx.post(
        'http://localhost:8080/api/v1/esm2/predict-mutation',
        json={
            'sequence': test_sequence,
            'position': pos,
            'mutant_residue': mut
        },
        timeout=60.0
    )

    if response.status_code == 200:
        impact = response.json()
        print(f'{wt}{pos+1}{mut}:')
        print(f'  Wild-type score: {impact["wild_type_score"]:.4f}')
        print(f'  Mutant score: {impact["mutant_score"]:.4f}')
        print(f'  Delta: {impact["delta_score"]:.4f}')
        print(f'  Effect: {impact["effect_classification"]}')
        print(f'  Confidence: {impact["confidence"]:.4f}\n')
    else:
        print(f'{wt}{pos+1}{mut}: Error {response.status_code}')

## Summary

You've successfully run the OpenBioDesign platform with real ML models!

### Remote Access
The localtunnel URLs from Cell 7 let you access the platform from **any device** on any network.
Share the frontend URL with collaborators to demo the platform.

### What We Demonstrated
1. **ESM2 Binding Site Detection**: Attention-based identification of functional residues
2. **Sequence Fitness Scoring**: Log-likelihood based protein stability assessment
3. **Binder Generation**: ML-powered candidate sequence generation
4. **Structure Prediction**: ESMFold 3D structure prediction with confidence scores
5. **Mutation Analysis**: Zero-shot prediction of mutation effects

### Resources
- [ESM2 Paper](https://proceedings.mlr.press/v162/esm2.html)
- [ESMFold Paper](https://www.biorxiv.org/content/10.1101/2022.07.20.500901v2)
- [OpenBioDesign](https://github.com/rager1904/OpenBioDesign)